In [459]:
def read_file_line_by_line(file_path):
    lines = []
    with open(file_path, 'r') as file:
        for line in file:
            if lines != "\n":
                lines.append(line.strip())  # Lưu từng dòng sau khi loại bỏ ký tự xuống dòng
    return lines

# Đường dẫn tới file
file_path = './file/IE.asn'

# Lấy nội dung các dòng từ file
lines = read_file_line_by_line(file_path)

In [460]:
import shutil, os
def delete_all_in_folder(folder_path):
    # Kiểm tra xem thư mục có tồn tại không
    if os.path.exists(folder_path):
        # Xóa tất cả nội dung trong thư mục
        shutil.rmtree(folder_path)
        # Tạo lại thư mục rỗng nếu cần thiết
        os.makedirs(folder_path)
        print(f"All contents in the folder '{folder_path}' have been deleted.")
    else:
        print(f"The folder '{folder_path}' does not exist.")

def delete_all_except(filepath, folder_path):
    # Duyệt qua tất cả các tập tin và thư mục trong folder_path
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        
        # Kiểm tra nếu filename không phải là file cần giữ lại
        if filename != filepath:
            # Kiểm tra nếu là một file và xóa nó
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            # Kiểm tra nếu là một thư mục và xóa nó
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)


folder_path = './ie'
delete_all_in_folder(folder_path)

All contents in the folder './ie' have been deleted.


In [461]:
import re

def Format(input_string):
    # only word and number
    if input_string.startswith("id-"):
        input_string = input_string.replace("id-", "")
    s = re.sub(r'[^a-zA-Z0-9]', '', input_string)
    return s[0].upper() + s[1:]

def find_Min_Max(s):
    s = re.sub("-", "", s)
    min = ""
    max = ""
    pattern_Min_Max = r'SIZE\((\d+)\.\.(\w+)\)\)'
    match = re.search(pattern_Min_Max, s)
    if match:
        min = match.group(1)  # Giá trị "1"
        max = match.group(2)  # Giá trị "abc"
    return min, max



In [462]:
#  check Ext
IsExt = []

In [463]:
# IE handle
def HandleIEofSequence(s):
    ie, type, ext, tag, min, max = "", "", False, "", "", ""
    
    if "iE-Extension" in s:
        s = re.sub(r'\{|\}', '', s)
        arr = s.split()
        ie = arr[0]
        # type = arr[1] + arr[2]
        type = arr[2]
        if "OPTIONAL" in arr[3]:
            tag = "OPTIONAL"
        return ie, type, ext, tag, min, max
    
    if "protocolIEs" in s:
        s = re.sub(r'\{|\}', '', s)
        arr = s.split()
        ie = arr[0]
        type = arr[2]
        if "OPTIONAL" in s:
            tag = "OPTIONAL"
        return ie, type, ext, tag, min, max

    if "CONTAINING" in s:
        arr = s.split()
        ie = arr[0]
        type = arr[1]
        return ie, type, ext, tag, min, max

    s = s.replace(",", "")
    arr = s.split()
    if len(arr) == 2:
        ie = arr[0]
        type = arr[1]
    if len(arr) == 3:
        ie = arr[0]
        type = arr[1]
        ext, tag, min, max = HandleSizeTag(arr[2])
    if len(arr) > 3:
        ext, tag, min, max = HandleSizeTag(arr[2])
        if "OPTIONAL" in arr[3]:
            tag = "OPTIONAL"
        if "OPTIONAL" in arr[3]:
            ext = True
        ie = arr[0]
        type = arr[1]

    return ie, type, ext, tag, min, max

def HandleSizeTag(s):
    ext, tag, min, max = False, "", "", ""
    if "OPTIONAL" in s:
        tag = "OPTIONAL"
    if "..." in s:
        ext = True
    # (2) -> 2
    match1 = re.search(r"\((\d+)\)", s)
    # aa(16, ...)
    match2 = re.search(r'\((\d+),\s+\.\.\.', s)
    match2_1 = re.search(r'\((\d+),', s)
    # (SIZE(22..32))
    match3 = re.search(r'\((\d+)\.\.(\d+)\)', s)
    # (0..255, …) -> 0, 255
    match4 = re.search(r'\((\d+)\.\.(\d+),', s)
    match5 = re.search(r'(\d+)\.\.(\w+[\w\-]*)', s)
    if match1:
        min = match1.group(1)
        max = match1.group(1)
    elif match2:
        min = match2.group(1)
        max = match2.group(1)
    elif match2_1:
        min = match2_1.group(1)
        max = match2_1.group(1)
    elif match3:
        min = match3.group(1)
        max = match3.group(2)
    elif match4:
        min = match4.group(1)
        max = match4.group(2)
    elif match5:
        min = match5.group(1)
        max = match5.group(2)
    
    return ext, tag, min, max

def Handle_NGAP_PROTOCOL_EXTENSION(s):
    cleaned_string = re.sub(r'[()\[\]{},]', ' ', s)
    arr = cleaned_string.split()
    ie = arr[1]
    type = arr[5]
    criticality = arr[3]
    present = arr[7]
    if len(arr) == 10:
        type = arr[7]
        present = arr[9]
    return ie, type, criticality, present

In [464]:
# IE handle for CHOICE
def Handle_CHOICE(s):
    ie, type, ext, tag, min, max = "", "", False, "", "", ""
    s = re.sub(r'\{|\}', '', s)
    s = s.replace(",", "")
    arr = s.split()
    ie = arr[0]
    if "choice-Extensions" in s and len(arr) == 3:
        # type = arr[1] + arr[2]
        type = arr[2]
    elif "BITSTRING" in s and len(arr) == 3:
        type = "BITSTRING"
        ext, tag, min, max = HandleSizeTag(arr[2])
    elif len(arr) == 2:
        type = arr[1]
    return ie, type, ext, tag, min, max



In [465]:
structs = {}
structSmall = {}
structsChoice = {}
structsEnumerate = {}
for i, line in enumerate(lines):

    if line == "" or line.startswith("--") or re.match(r'^\.\.\.', line) or line == "}": continue

    elif line.endswith("{"):
        if not "SEQUENCE" in line and not "NGAP-PROTOCOL-EXTENSION ::= {" in line \
            and not "NGAP-PROTOCOL-IES ::= {" in line and not "::= CHOICE {" in line \
                and not "::= ENUMERATED {" in line:
            print(line)
        
        # sequence
        if "SEQUENCE" in line:
            info = line.split()
            structs[info[0]] = {}
            structs[info[0]]["ies"] = []
            structs[info[0]]["ext"] = False
            j=i+1
            while j<len(lines):
                if lines[j] == "" or lines[j].startswith("--"):
                    j+=1
                if "..." in lines[j]:
                    structs[info[0]]["ext"] = True
                    j += 1
                if lines[j] == "}": break
                ie, type, ext, tag, min, max = "", "", False, "", "", ""
                ie, type, ext, tag, min, max = HandleIEofSequence(lines[j])
                structs[info[0]]["ies"].append({
                    "ie": ie,
                    "type": type,
                    "ext": ext,
                    "tag": tag,
                    "min": min,
                    "max": max
                })
                if ie == "iE-Extensions":
                    IsExt.append(info[0])
                j += 1
            if j>=len(lines): break
        if "NGAP-PROTOCOL-EXTENSION ::= {" in line:
            info = line.split()
            structs[info[0]] = {}
            structs[info[0]]["ies"] = []
            structs[info[0]]["ext"] = False
            structs[info[0]]["choice"] = False
            if "..." in lines[i+1]:
                structs[info[0]]["ext"] = True
            else:
                j=i+1
                while j<len(lines):
                    if lines[j] == "" or lines[j].startswith("--"):
                        j+=1
                    if "..." in lines[j]:
                        structs[info[0]]["ext"] = True
                        j += 1
                    if lines[j] == "}": break
                    else: structs[info[0]]["choice"] = True
                    if structs[info[0]]["choice"] == False and "|" in lines[j]:
                        structs[info[0]]["choice"] = True
                    ie, type, criticality, present = Handle_NGAP_PROTOCOL_EXTENSION(lines[j])
                    structs[info[0]]["ies"].append({
                        "ie": ie,
                        "type": type,
                        "criticality": criticality,
                        "present": present
                    })
                    if ie == "iE-Extensions":
                        IsExt.append(info[0])
                    j += 1
                if j>=len(lines): break
        if "NGAP-PROTOCOL-IES ::= {" in line:
            info = line.split()
            structs[info[0]] = {}
            structs[info[0]]["ies"] = []
            structs[info[0]]["ext"] = False
            structs[info[0]]["choice"] = False
            if "..." in lines[i+1]:
                structs[info[0]]["ext"] = True
            else:
                j=i+1
                while j<len(lines):
                    if lines[j] == "":
                        j+=1
                    if "..." in lines[j]:
                        structs[info[0]]["ext"] = True
                        j += 1
                    if lines[j] == "}": break
                    else: structs[info[0]]["choice"] = True
                    if structs[info[0]]["choice"] == False and "|" in lines[j]:
                        structs[info[0]]["choice"] = True
                    
                    ie, type, criticality, present = Handle_NGAP_PROTOCOL_EXTENSION(lines[j])
                    structs[info[0]]["ies"].append({
                        "ie": ie,
                        "type": type,
                        "criticality": criticality,
                        "present": present
                    })
                    if ie == "iE-Extensions":
                        IsExt.append(info[0])
                    j += 1
                if j>=len(lines): break
        if "::= CHOICE {" in line:
            info = line.split()
            structsChoice[info[0]] = {}
            structsChoice[info[0]]["ies"] = []
            structsChoice[info[0]]["ext"] = False
            if "..." in lines[i+1]:
                structsChoice[info[0]]["ext"] = True
            else:
                j = i + 1
                while j < len(lines):
                    if lines[j] == "":
                        j+=1
                    if "..." in lines[j]:
                        structsChoice[info[0]]["ext"] = True
                        j += 1
                    if lines[j] == "}": break
                    
                    ie, type, ext, tag, min, max = "", "", False, "", "", ""
                    ie, type, ext, tag, min, max = Handle_CHOICE(lines[j])
                    structsChoice[info[0]]["ies"].append({
                        "ie": ie,
                        "type": type,
                        "ext": ext,
                        "tag": tag,
                        "min": min,
                        "max": max
                    })
                    j += 1
                if ie == "iE-Extensions":
                    IsExt.append(info[0])
                if j>=len(lines): break
        if "::= ENUMERATED {" in line:
            info = line.split()
            structsEnumerate[info[0]] = {}
            structsEnumerate[info[0]]["ies"] = []
            structsEnumerate[info[0]]["ext"] = False
            if "..." in lines[i+1]:
                structsEnumerate[info[0]]["ext"] = True
            else:
                j = i + 1
                while j < len(lines):
                    if lines[j] == "":
                        j+=1
                    if "..." in lines[j]:
                        structsEnumerate[info[0]]["ext"] = True
                        structsEnumerate[info[0]]["ies"].append(lines[j].replace(",", ""))
                        j += 1
                    if lines[j] == "}": break
                    structsEnumerate[info[0]]["ies"].append(lines[j].replace(",", ""))
                    j += 1
                if j>=len(lines): break
    elif not "ProtocolExtensionContainer" in line and not "ProtocolIE-SingleContainer" in line \
        and not "{ ID id-" in line and not line.endswith(",") and not len(line.split()) == 1:
        # print(line)
        arr = line.split()
        
        if len(arr) == 3:
            structSmall[arr[0]] = {}
            structSmall[arr[0]]["type"] = arr[2]
        if len(arr) == 4: # BITSTRING, INTEGER, OCTETSTRING
            structSmall[arr[0]] = {}
            structSmall[arr[0]]["type"] = arr[2]
            _, _, min, max = HandleSizeTag(arr[3])
            structSmall[arr[0]]["ext"] = False
            structSmall[arr[0]]["min"] = min
            structSmall[arr[0]]["max"] = max
        if len(arr) == 5: # PrintableString, BITSTRING, INTEGER
            structSmall[arr[0]] = {}
            structSmall[arr[0]]["type"] = arr[2]
            _, _, min, max = HandleSizeTag(arr[3])
            if "..." in line: structSmall[arr[0]]["ext"] = True
            structSmall[arr[0]]["min"] = min
            structSmall[arr[0]]["max"] = max
        if len(arr) == 6: # " OF "
            # if arr[0] == "AllowedNSSAI":
            #     print(line)
            structSmall[arr[0]] = {}
            structSmall[arr[0]]["type"] = arr[5]
            _, _, min, max = HandleSizeTag(arr[3])
            structSmall[arr[0]]["ext"] = True
            if "..." in line: structSmall[arr[0]]["ext"] = True
            structSmall[arr[0]]["min"] = min
            structSmall[arr[0]]["max"] = max
        if len(arr) > 6: # " ENUMERATED "
            structSmall[arr[0]] = {}
            structSmall[arr[0]]["type"] = arr[2]
            if "..." in line: structSmall[arr[0]]["ext"] = True
            s = re.sub(r'\{|\}', '', line)
            s = s.replace(",", "")
            arr = s.split()
            # print(arr)


        # if "..." in line:
        #     structSmall[arr[0]]["ext"] = True
        # if len(arr) == 5 and "..." in arr[4]: pass
        #     type = arr[2]
        #     structSmall[arr[0]]["ext"] = False
        #     _, _, min, max = HandleSizeTag(arr[3])
        # structSmall[arr[0]]["type"] = type
        # structSmall[arr[0]]["min"] = min
        # structSmall[arr[0]]["max"] = max

In [466]:
# PDU 

file_path = './file/pdu.asn'
# Lấy nội dung các dòng từ file
lines = read_file_line_by_line(file_path)

structPDU_SEQUENCE = {}
structPDU_IES = {}
structPDU_PRIVATE_IES = {}

for i, line in enumerate(lines):

    if line == "" or line.startswith("--") or re.match(r'^\.\.\.', line) or line == "}": continue

    if line.endswith("{"):
        # if not "SEQUENCE" in line and not "NGAP-PROTOCOL-EXTENSION ::= {" in line and not "NGAP-PROTOCOL-IES ::= {" in line and not "::= CHOICE {" in line:
        #     print(line)
        
        # sequence
        if "SEQUENCE" in line:
            info = line.split()
            structPDU_SEQUENCE[info[0]] = {}
            structPDU_SEQUENCE[info[0]]["ies"] = []
            structPDU_SEQUENCE[info[0]]["ext"] = False
            j=i+1
            while j<len(lines):
                if lines[j] == "" or lines[j].startswith("--"):
                    j+=1
                if "..." in lines[j]:
                    structPDU_SEQUENCE[info[0]]["ext"] = True
                    j += 1
                if lines[j] == "}": break
                arr = lines[j].split()
                cleaned_string = re.sub(r'[()\[\]{},]', ' ', arr[2])
                structPDU_SEQUENCE[info[0]]["ies"].append({
                    "ie": arr[0],
                    "type": cleaned_string,
                })
                j += 1
        
        if "NGAP-PROTOCOL-IES ::= {" in line:
            info = line.split()
            structPDU_IES[info[0]] = {}
            structPDU_IES[info[0]]["ies"] = []
            structPDU_IES[info[0]]["ext"] = False
            structPDU_IES[info[0]]["choice"] = True
            if "..." in lines[i+1]:
                structPDU_IES[info[0]]["ext"] = True
            else:
                j=i+1
                while j<len(lines):
                    if lines[j] == "":
                        j+=1
                    if "..." in lines[j]:
                        structPDU_IES[info[0]]["ext"] = True
                        j += 1
                    # if structPDU_IES[info[0]]["choice"] == False and "|" in lines[j]:
                    #     structPDU_IES[info[0]]["choice"] = True
                    if lines[j] == "}": break

                    ie, type, criticality, present = Handle_NGAP_PROTOCOL_EXTENSION(lines[j])
                    structPDU_IES[info[0]]["ies"].append({
                        "ie": ie,
                        "type": type,
                        "criticality": criticality,
                        "present": present
                    })
                    j += 1
        
        if "NGAP-PRIVATE-IES ::= {" in line:
            info = line.split()
            structPDU_PRIVATE_IES[info[0]] = {}
            structPDU_PRIVATE_IES[info[0]]["ies"] = []
            structPDU_PRIVATE_IES[info[0]]["ext"] = False
            if "..." in lines[i+1]:
                structPDU_PRIVATE_IES[info[0]]["ext"] = True
            else:
                j=i+1
                while j<len(lines):
                    if lines[j] == "":
                        j+=1
                    if "..." in lines[j]:
                        structPDU_PRIVATE_IES[info[0]]["ext"] = True
                        j += 1
                    if lines[j] == "}": break

                    structPDU_PRIVATE_IES[info[0]]["ies"].append({
                        "ie": "",
                        "type": "",
                        "criticality": "",
                        "present": ""
                    })
                    j += 1
        


In [467]:
# structs
# structsEnumerate
# structSmall

# structPDU_SEQUENCE
# structPDU_IES
# structPDU_PRIVATE_IES

In [468]:
# PrivateMessage ::= SEQUENCE {
#  privateIEs PrivateIE-Container {{PrivateMessageIEs}},
#  ...
# }
pduSequence = {}
pduSendIes = {}

file_path = './file/pdu.asn'
# Lấy nội dung các dòng từ file
lines = read_file_line_by_line(file_path)

for i, line in enumerate(lines):
    if line == "" or line.startswith("--") or re.match(r'^\.\.\.', line) or line == "}": continue
    # ::= SEQUENCE {
    if line.endswith("::= SEQUENCE {"):
        arr = lines[i].split()
        name = arr[0]
        pduSequence[name] = {}
        arr = lines[i+1].replace("{", "").replace("}", "").split()
        pduSequence[name]["ie"] = arr[0]
        pduSequence[name]["type"] = arr[2]
        pduSequence[name]["ext"] = True
        IsExt.append(name)
        
    # NGAP-PROTOCOL-IES ::= {
    if line.endswith("NGAP-PROTOCOL-IES ::= {"):
        info = line.split()
        pduSendIes[info[0]] = {}
        pduSendIes[info[0]]["ies"] = []
        pduSendIes[info[0]]["ext"] = False
        if "..." in lines[i+1]:
            pduSendIes[info[0]]["ext"] = True
        else:
            j = i + 1
            while j < len(lines):
                if "..." in lines[j]:
                    pduSendIes[info[0]]["ext"] = True
                    j += 1
                elif lines[j] == "}": break
                else:
                    arr = lines[j].replace("{", "").replace("}", "").replace("|", "").replace(",", "").split() # == 8
                    ie, type, criticality, present = Handle_NGAP_PROTOCOL_EXTENSION(lines[j])
                    pduSendIes[info[0]]["ies"].append({
                        "ie": ie,
                        "type": type,
                        "criticality": criticality,
                        "present": present
                    })
                    j += 1


In [469]:
# PROCEDURE CODE - CRITICALITY

structSend = {}

file_path = './file/Interface_Elementary_Procedures.asn'
# Lấy nội dung các dòng từ file
lines = read_file_line_by_line(file_path)

for i, line in enumerate(lines):
    if line == "" or line.startswith("--") or re.match(r'^\.\.\.', line) or line == "}": continue
    if line.endswith("NGAP-ELEMENTARY-PROCEDURE ::= {"):
        info = line.split()
        structSend[info[0]] = {}
        structSend[info[0]]["INITIATINGMESSAGE"] = ""
        structSend[info[0]]["SUCCESSFULOUTCOME"] = ""
        structSend[info[0]]["UNSUCCESSFULOUTCOME"] = ""
        structSend[info[0]]["PROCEDURECODE"] = ""
        structSend[info[0]]["CRITICALITY"] = ""

        j = i + 1
        while j < len(lines):
            if "INITIATINGMESSAGE" in lines[j]:
                structSend[info[0]]["INITIATINGMESSAGE"] = lines[j].split()[1]
            if "SUCCESSFULOUTCOME" in lines[j]:
                structSend[info[0]]["SUCCESSFULOUTCOME"] = lines[j].split()[1]
            if "UNSUCCESSFULOUTCOME" in lines[j]:
                structSend[info[0]]["UNSUCCESSFULOUTCOME"] = lines[j].split()[1]
            if "PROCEDURECODE" in lines[j]:
                structSend[info[0]]["PROCEDURECODE"] = lines[j].split()[1]
            if "CRITICALITY" in lines[j]:
                structSend[info[0]]["CRITICALITY"] = lines[j].split()[1]
            if "}" in lines[j]: break
            j += 1


In [470]:
# write to file

dir = "./ie/"

def write_file(file, string):
    file = Format(file)
    file_path = dir + file + ".go"
    if os.path.exists(file_path):
        print(f"File {file_path} đã tồn tại. Thoát chương trình.")
        return 
    else:
        with open(file_path, 'w') as go_file:
            go_file.write(string)

def GetType(s):
    if s == "INTEGER": return "aper.Integer"
    if s == "BITSTRING": return "aper.BitString"
    if s == "OCTETSTRING": return "aper.OctetString"
    if s == "ENUMERATED": return "aper.Enumerated"
    if s == "PrintableString": return "aper.OctetString"
    return s

In [471]:
# structSmall
for i, name in enumerate(structSmall):
    write = "package ie\nimport \"gen/aper\"\n"
    # print(i, name, structSmall[name])
    if "min" in structSmall[name] and "ext" in structSmall[name]:
        write += "type "+ Format(name) + " struct {\nValue " + GetType(Format(structSmall[name]["type"])) +\
            "`" + str(structSmall[name]["ext"]) + "," + structSmall[name]["min"] + "," + structSmall[name]["max"] + "`\n}\n"
    elif "ext" in structSmall[name] and not "min" in structSmall[name]:
        write += "type "+ Format(name) + " struct {\nValue " + GetType(Format(structSmall[name]["type"])) +\
            "`" + str(structSmall[name]["ext"]) + "`\n}"
    else: write += "type "+ Format(name) + " struct {\nValue " + GetType(Format(structSmall[name]["type"])) + "\n}\n"
    write_file(name, write)

In [472]:
# structsChoice

for i, name in enumerate(structsChoice):
    write = "package ie\nimport \"gen/aper\"\n"
    write += "type "+ Format(name) + " struct {\n"
    for ie in structsChoice[name]["ies"]:
        if ie["type"] in IsExt: ie["ext"] = True
        write += Format(ie["ie"]) + " " + GetType(Format(ie["type"])) + " `" +\
            str(ie["ext"]) + "," + str(ie["tag"]) + "`\n"
    write += "}\n"
    write_file(name, write)

In [473]:
# structsEnumerate

def count_elements_before(arr):
    count = 0
    for element in arr:
        if element == "...":
            break
        count += 1
    return count - 1

for i, name in enumerate(structsEnumerate):
    if name == "ExpectedHOInterval":
        write = "package ie\nimport \"gen/aper\"\n"
        write += "type "+ Format(name) + " struct {\nValue aper.Enumerated " +\
            "`" + str(structsEnumerate[name]["ext"]) + ",0,6`\n}\n"
        write_file(name, write)
    else:
        write = "package ie\nimport \"gen/aper\"\n"
        write += "type "+ Format(name) + " struct {\nValue aper.Enumerated " +\
            "`" + str(structsEnumerate[name]["ext"]) + ",0," + str(count_elements_before(structsEnumerate[name]["ies"])) + "`\n}\n"
        write_file(name, write)

In [474]:
# structs

for i, name in enumerate(structs):
    write = "package ie\nimport \"gen/aper\"\n"
    # print(i, name, structs[name])
    write += "type "+ Format(name) + " struct {\n"
    for ie in structs[name]["ies"]:
        # print(ie)
        if "ext" in ie:
            if ie["type"] in IsExt: ie["ext"] = True
            write += Format(ie["ie"]) + " " + GetType(Format(ie["type"])) + " `" +\
                str(ie["ext"]) + "," + str(ie["tag"]) + "`\n"
        else:
            write += Format(ie["ie"]) + " " + GetType(Format(ie["type"])) + " `" +\
                "," + str(ie["criticality"]) + "," + str(ie["present"]) + "`\n"
    write += "}\n"
    write_file(name, write)

In [476]:
# Criticality		::= ENUMERATED { reject, ignore, notify }
# Presence		::= ENUMERATED { optional, conditional, mandatory }

# PrivateIE-ID	::= CHOICE {
# 	local				INTEGER (0..65535),
# 	global				OBJECT IDENTIFIER
# }

# ProcedureCode		::= INTEGER (0..255)
# ProtocolExtensionID	::= INTEGER (0..65535)
# ProtocolIE-ID		::= INTEGER (0..65535)
# TriggeringMessage	::= ENUMERATED { initiating-message, successful-outcome, unsuccessfull-outcome }

write = "package ie\nimport \"gen/aper\"\n"
write += "type Criticality struct {\nValue aper.Enumerated `aper:\"valueLB:0,valueUB:2\"`\n}\n"
write += "type ProcedureCode struct {\nValue aper.Integer `aper:\"valueLB:0,valueUB:255\"`\n}\n"
write += "type TriggeringMessage struct {\nValue aper.Enumerated `aper:\"valueLB:0,valueUB:2\"`\n}\n"
write += "type ProtocolIEID struct {\nValue aper.Integer `aper:\"valueLB:0,valueUB:65535\"`\n}\n"
write_file("common", write)

In [477]:
for i, name in enumerate(pduSequence):
    write = "package ie\nimport \"gen/aper\"\n"
    struct = pduSequence[name]
    write += "type "+ Format(name) + " struct {\n"
    if struct["type"] in IsExt: struct["ext"] = True
    write += Format(struct["ie"]) + " " + GetType(Format(struct["type"])) + " `" +\
        str(struct["ext"]) + "`\n"
    write += "}\n"
    write_file(name, write)

In [478]:
for i, name in enumerate(pduSendIes):
    write = "package ie\nimport \"gen/aper\"\n" + "type "+ Format(name) + " struct {\n"
    for ie in pduSendIes[name]["ies"]:
        write += Format(ie["ie"]) + " " + GetType(Format(ie["type"])) + " `" +\
            "," + str(ie["criticality"]) + "," + str(ie["present"]) + "`\n"
    write += "}\n"
    write_file(name, write)

# PrivateMessageIEs NGAP-PRIVATE-IES ::= {
# 	...
# }
write = "package ie\nimport \"gen/aper\"\n" + "type PrivateMessageIEs struct {\n}" + ""
write_file("PrivateMessageIEs", write)